GEETA 4B — worksheet splitting + deterministic pattern tutoring.

Found live-testing (2026-10-13): a photo of a whole worksheet page became ONE
"question". The verified plan only ever covered pattern 1, so asking for help
on question 4 got "Next term in pattern 1", and the model filled the gap with
invented advice ("hop by 10") for a pattern that jumps by 50.

This module:
  1. splits an OCR'd page into its numbered questions (+ "write the number" word items),
  2. lets the kid pick one ("question 4", "the fourth one", "no 4"),
  3. tutors number patterns and number-words with Python-computed hints only:
     find the jump -> fill each gap -> concrete counting anchor. Every number
     said comes from the kid's own page; a missing number is never said first.


## Bootstrap

This notebook is a GEETA runtime module. Executed standalone (e.g. `jupyter execute tools/verifier.ipynb`),
the next cell registers the notebook-as-module loader so `geeta.*` imports resolve to `tools/*.ipynb`.
When imported as `geeta.<name>` via `tools/_nbmodule.ipynb`, the same cell runs (idempotent).


In [ ]:
# --- GEETA notebook-module bootstrap (idempotent; makes tools/*.ipynb importable as geeta.*) ---
import json as _json
import pathlib as _pathlib

_ROOT = _pathlib.Path.cwd()
if not (_ROOT / "tools" / "_nbmodule.ipynb").exists():      # run from project root or tools/
    _ROOT = _ROOT.parent
_nb = _json.loads((_ROOT / "tools" / "_nbmodule.ipynb").read_text())
_src = "".join("".join(c["source"]) for c in _nb["cells"] if c["cell_type"] == "code")
exec(compile(_src, str(_ROOT / "tools" / "_nbmodule.ipynb"), "exec"), globals())
bootstrap(_ROOT)  # noqa: E501


In [ ]:

from __future__ import annotations

import random
import re
from dataclasses import dataclass, field
from typing import Optional

# ---------------------------------------------------------------------------
# Worksheet splitting
# ---------------------------------------------------------------------------

_ITEM_MARK = re.compile(r"(?:(?<=\s)|^)(?:(?:Q|Qn|Question|QUESTION|question)\s*(\d{1,2})\s*[.):]?|\(?(\d{1,2})\s*[.)])\s+")


def _mk(m) -> str:
    """The question number of a marker: '3.' / '3)' / 'Q3' / 'Question 3:'."""
    return m.group(1) or m.group(2)
_BLANK = re.compile(r"_+|\.{3,}|\?|□|\[\s*\]")

_ONES = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
         "seven": 7, "eight": 8, "nine": 9}
_TEENS = {"ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14,
          "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19}
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60,
         "seventy": 70, "eighty": 80, "ninety": 90}
_ONES_RE = "|".join(_ONES)
_WORD_NUM = re.compile(
    r"\b(" + _ONES_RE + r")\s+hundred(?:\s+and\s+(?:"
    r"(?:" + "|".join(_TENS) + r")(?:[\s-]+(?:" + _ONES_RE + r"))?"
    r"|" + "|".join(_TEENS) + r"|" + _ONES_RE + r"))?\b", re.I)


def words_to_int(phrase: str) -> Optional[int]:
    """'four hundred and eighty-five' -> 485 (only the 100-999 shape we split on)."""
    toks = re.findall(r"[a-z]+", phrase.lower())
    total, cur = 0, 0
    for t in toks:
        if t == "and":
            continue
        if t in _ONES:
            cur += _ONES[t]
        elif t in _TEENS:
            cur += _TEENS[t]
        elif t in _TENS:
            cur += _TENS[t]
        elif t == "hundred":
            cur = (cur or 1) * 100
        else:
            return None
    total += cur
    return total or None


@dataclass
class WorksheetItem:
    label: str                 # what the kid calls it: "4", "word 2"
    kind: str                  # "pattern" | "words" | "other"
    text: str                  # the question as the kid should see it
    terms: list = field(default_factory=list)   # pattern: ints and None for gaps
    step: Optional[int] = None
    answer: Optional[int] = None                # words: the digits answer
    suspect: dict = field(default_factory=dict)  # pattern: {position: number that doesn't fit}
    meta: dict = field(default_factory=dict)     # arith: {"problem": geeta.arith.Problem}

    @property
    def gaps(self) -> list[int]:
        return [i for i, v in enumerate(self.terms) if v is None]

    @property
    def finished_on_paper(self) -> bool:
        return self.kind == "pattern" and not self.gaps


def _parse_pattern(body: str) -> tuple[Optional[list], str]:
    """'350, , , 500, 550, , 650 Write the ...' -> ([350,None,None,500,550,None,650], rest).
    Returns (None, body) when the text isn't a comma-separated number pattern."""
    body = _BLANK.sub(" ", body)
    m = re.match(r"\s*((?:\d[\d ]*)?(?:\s*,\s*(?:\d+)?)+)", body)
    if not m:
        return None, body
    seq, rest = m.group(1), body[m.end():]
    # a trailing gap is written as "900, , " - drop only the final empty after
    # the last comma if the OCR didn't leave a blank there
    toks = [t.strip() for t in seq.split(",")]
    terms = []
    for t in toks:
        if t == "":
            terms.append(None)
        elif re.fullmatch(r"\d+", t):
            terms.append(int(t))
        else:
            return None, body
    while terms and terms[-1] is None and not seq.rstrip().endswith(","):
        terms.pop()
    if seq.rstrip().endswith(",") and (not terms or terms[-1] is not None):
        terms.append(None)
    if sum(1 for v in terms if v is not None) < 2 or len(terms) < 3:
        return None, body
    return terms, rest


def pattern_step(terms: list) -> Optional[int]:
    """The constant jump of an arithmetic pattern, checked against EVERY known
    term. None if the known numbers don't fit one jump (OCR slip / not a
    simple pattern) - we then never guess."""
    known = [(i, v) for i, v in enumerate(terms) if v is not None]
    if len(known) < 2:
        return None
    (i0, v0), (i1, v1) = known[0], known[1]
    if (v1 - v0) % (i1 - i0):
        return None
    step = (v1 - v0) // (i1 - i0)
    if step == 0:
        return None
    if any(v != v0 + (i - i0) * step for i, v in known):
        return None
    return step


def fit_with_one_slip(terms: list) -> tuple[Optional[int], dict]:
    """A filled-in pattern where exactly ONE number breaks the jump (the kid's
    slip, or our misread of their handwriting): return (step, {pos: odd number})."""
    known = [(i, v) for i, v in enumerate(terms) if v is not None]
    if len(known) < 4:
        return None, {}
    diffs = {}
    for (i, a), (j, b) in zip(known, known[1:]):
        if (b - a) % (j - i) == 0:
            d = (b - a) // (j - i)
            diffs[d] = diffs.get(d, 0) + 1
    if not diffs:
        return None, {}
    step = max(diffs, key=diffs.get)
    if step == 0:
        return None, {}
    for anchor_i, anchor_v in known:
        odd = {i: v for i, v in known if v != anchor_v + (i - anchor_i) * step}
        if len(odd) == 1:
            return step, odd
    return None, {}


def fill_pattern(terms: list, step: int) -> list[int]:
    i0, v0 = next((i, v) for i, v in enumerate(terms) if v is not None)
    return [v0 + (i - i0) * step for i in range(len(terms))]


def _maybe_arith(item: WorksheetItem) -> WorksheetItem:
    """A non-pattern question Python can read exactly (27 + 15, word problems...)."""
    from geeta.arith import parse_question
    p = parse_question(item.text)
    if p is not None:
        item.kind = "arith"
        item.answer = p.answer if isinstance(p.answer, int) else None
        item.meta = {"problem": p}
    return item


# "The number sentence:" / "The answer: ____" lines are answer spaces, not part of the question
_BOILER = re.compile(r"\b(?:the )?(?:number sentence|answer|working(?: out)?)\s*:\s*(?:_+)?", re.I)
# a numbered TIP / STEPS box ("1. Read the problem carefully. 2. Look for clue words...")
_INSTRUCTION = re.compile(r"^\s*(?:read|look|underline|remember|write|show|check|draw|circle|colou?r|use|think|"
                          r"trace|join|match|label|tick|cross out|then|cut|stick|"
                          r"explain|work|try|make sure|don'?t|do not|always|ask|find the clue|highlight)\b", re.I)


_PICTURE_TASK = re.compile(r"^\s*(?:draw|trace|colou?r|circle|tick|join|match|cut|stick|shade)\b", re.I)


def _near_empty(body: str) -> bool:
    """'' or 'segments angles' - the question is a picture the text can't show."""
    return not re.sub(r"\b(?:segments?|angles?|sides?|corners?|vertices|answer|name)\b|[\s:_.,]", "", body or "", flags=re.I)


def _is_heading(body: str) -> bool:
    """'Trace the parallel lines with a blue crayon. Then use a red crayon...' - an instruction for a group."""
    sents = [s for s in re.split(r"(?<=[.!:])\s+", (body or "").strip()) if s.strip()]
    return bool(sents) and "?" not in body and all(_INSTRUCTION.search(s) for s in sents)


def _last_heading(pre: str) -> str:
    sents = [s.strip() for s in re.split(r"(?<=[.!:])\s+", re.sub(r"\s+", " ", pre or "")) if s.strip()]
    for s in reversed(sents):
        s = re.sub(r"^\d+\s+", "", s)                  # "Page: 9 Draw lines of symmetry." -> drop the 9
        if _INSTRUCTION.search(s) and "?" not in s:
            return s
    return ""


def _questionish(body: str) -> bool:
    """Is this numbered line a question to answer (not a tip like 'Read the problem carefully')?"""
    b = _BOILER.sub(" ", body).strip()
    if not b:
        return False
    if "?" in b or re.search(r"_{2,}|□|\[\s*\]", b) or re.search(r"\d", b) or _WORD_NUM.search(b):
        return True
    return not _INSTRUCTION.search(b) and len(b.split()) <= 4      # "2. cat, hat, ___" style short items


def _marker_run(text: str) -> list:
    """The best run of question markers that count up by one - starting at ANY
    number (a page can start at question 21). A stray '9.' can't fake a run.
    Runs are ranked by how many of their items are real QUESTIONS, so a numbered
    'The Steps: 1. Read... 4. Remember...' tip box never beats questions 1-3 below it."""
    cands = list(_ITEM_MARK.finditer(text))
    best: list = []
    best_key = (-1, -1)
    for i, first in enumerate(cands):
        run, want = [first], int(_mk(first)) + 1
        for m in cands[i + 1:]:
            if int(_mk(m)) == want:
                run.append(m)
                want += 1
        score = 0
        for m in run:
            nxt = next((c.start() for c in cands if c.start() > m.start()), len(text))
            body = text[m.end():nxt]
            # a line with a gap to fill (___, ?, r __) is a real question; a filled-in
            # "12 ÷ 3 = 4" is usually a worked example
            score += 2 * bool(re.search(r"_{2,}|\?|□|\[\s*\]", body)) + _questionish(body)
        key = (score, len(run))
        if key > best_key:                       # ties keep the earlier run
            best, best_key = run, key
    if len(best) == 1 and int(_mk(best[0])) != 1:
        return []                     # a lone "7." isn't a worksheet
    return best


_SUB_MARK = re.compile(r"\(([a-h])\)\s*")


def _item_from(label: str, body: str) -> tuple[WorksheetItem, str]:
    """One question (or sub-question) -> (item, leftover text after a pattern)."""
    terms, rest = _parse_pattern(body)
    if terms is None:
        # "The pattern is 1, 11, 21, 31, ... what comes next?" - the pattern starts mid-sentence
        clean = _BLANK.sub(" ", body.replace("…", " "))
        m0 = None if re.search(r"\d+(?:st|nd|rd|th) (?:one|number|term|step)", body, re.I) else \
            re.search(r"\d[\d ]*\s*,", clean)
        if m0:
            terms, rest = _parse_pattern(clean[m0.start():])
            if terms is not None and None not in terms and re.search(r"\bnext\b|comes after|what comes", body, re.I):
                terms = terms + [None]
    if terms is not None:
        step = pattern_step(terms)
        suspect = {}
        if step is None:
            step, suspect = fit_with_one_slip(terms)
            if step is not None:
                terms = [None if i in suspect else v for i, v in enumerate(terms)]
        if step is not None:
            shown = ", ".join("___" if v is None else str(v) for v in terms)
            return WorksheetItem(label, "pattern", shown, terms, step, suspect=suspect), rest
    item = _maybe_arith(WorksheetItem(label, "other", body))
    if item.kind == "other" and _PICTURE_TASK.search(body):
        item.kind = "picture"
        item.meta["heading"] = body.rstrip(":").strip()
    return item, ""


def split_worksheet(text: str) -> list[WorksheetItem]:
    """Split an OCR'd page into separate questions (and (a)/(b) sub-questions)."""
    text = re.sub(r"\s+", " ", text or "").strip()
    marks = _marker_run(text)
    items: list[WorksheetItem] = []
    leftovers: list[str] = []
    if not marks:
        # a typed single pattern: "what comes next? 2, 4, 6, __"
        text = text.replace("…", " ")             # "1, 11, 21, 31, … what comes next?" - the pattern goes on
        m0 = None if re.search(r"\d+(?:st|nd|rd|th) (?:one|number|term|step)", text, re.I) else \
            re.search(r"\d[\d ]*\s*,", _BLANK.sub(" ", text))     # "the 10th one" isn't "what comes next"
        terms, rest = _parse_pattern(_BLANK.sub(" ", text)[m0.start():]) if m0 else (None, text)
        step = pattern_step(terms) if terms else None
        if step is not None and None not in terms and re.search(r"\bnext\b|comes after|what comes", text, re.I):
            terms = terms + [None]                 # "5, 10, 15, what comes next?"
        if step is not None and None in terms:
            shown = ", ".join("___" if v is None else str(v) for v in terms)
            items.append(WorksheetItem("typed", "pattern", shown, terms, step))
        else:
            leftovers.append(text)
    bodies = []
    for k, m in enumerate(marks):
        end = marks[k + 1].start() if k + 1 < len(marks) else len(text)
        body = re.sub(r"\s+", " ", _BOILER.sub(" ", text[m.end():end])).strip()
        if k + 1 == len(marks):
            body = re.split(r"\b(?:end of (?:the )?test|my score|total score|score:)\b", body, flags=re.I)[0].strip()
        if "?" in body and not _SUB_MARK.search(body):
            # after the last "?" is the answer line ("Flo should take ___") or a kid's pencil
            # working ("9/4") - neither is part of the question
            tail = body[body.rfind("?") + 1:]
            # ...but keep answer CHOICES after it ("HORSE BEE PARROT LION APPLE")
            if not re.search(r"\d\s*,|_{2,}|□", tail) and len(re.findall(r"\b[A-Z]{2,}\b", tail)) < 2 \
                    and len(tail.split()) <= 8:
                body = body[:body.rfind("?") + 1]
        bodies.append(body)
    heading = _last_heading(text[:marks[0].start()]) if marks else ""
    for k, m in enumerate(marks):
        body = bodies[k]
        label = _mk(m)
        nxt_empty = k + 1 < len(bodies) and _near_empty(bodies[k + 1])
        if _near_empty(body) or (nxt_empty and _is_heading(body)):
            # a picture question (lines of symmetry, count the sides...): its words are the
            # heading above it; a heading sitting in THIS slot belongs to the next questions
            items.append(WorksheetItem(label, "picture", heading or "Look at the picture on your page.",
                                       meta={"heading": heading, "asks": body if _near_empty(body) else ""}))
            if not _near_empty(body):
                heading = body
            continue
        subs = list(_SUB_MARK.finditer(body))
        letters = [x.group(1) for x in subs]
        if len(subs) >= 2 and letters == [chr(ord("a") + i) for i in range(len(letters))]:
            stem = body[:subs[0].start()].strip()
            for j, sm in enumerate(subs):
                sub_end = subs[j + 1].start() if j + 1 < len(subs) else len(body)
                sub_body = body[sm.end():sub_end].strip()
                item, rest = _item_from(f"{label}{sm.group(1)}", sub_body)
                if item.kind == "other":
                    # re-read with the shared question stem ("What letter should come next...")
                    item, rest = _item_from(f"{label}{sm.group(1)}", f"{stem} {sub_body}")
                item.meta.setdefault("stem", stem)
                items.append(item)
                if rest.strip():
                    leftovers.append(rest)
            continue
        item, rest = _item_from(label, body)
        items.append(item)
        if rest.strip():
            leftovers.append(rest)
    # "Write the following numbers. four hundred and eighty-five ..." sections
    wn = 0
    for chunk in leftovers:
        for m in _WORD_NUM.finditer(chunk):
            val = words_to_int(m.group(0))
            if val is None:
                continue
            wn += 1
            items.append(WorksheetItem(f"word {wn}", "words", m.group(0).lower(),
                                       answer=val))
    return items



In [ ]:
# ---------------------------------------------------------------------------
# Talking to the kid about the page
# ---------------------------------------------------------------------------

_ORDINALS = {"first": 1, "second": 2, "third": 3, "fourth": 4, "fifth": 5,
             "sixth": 6, "seventh": 7, "eighth": 8, "ninth": 9, "tenth": 10,
             "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
             "seven": 7, "eight": 8, "nine": 9, "ten": 10}
_SELECT = re.compile(
    r"\b(?:question|questions|q|qn|no\.?|number|num|pattern|sum)\s*#?\s*(\d{1,2}|"
    + "|".join(_ORDINALS) + r")\b(?!\s*(?:million|thousand|hundred)\b)"   # "question 24 million" is an answer
    r"|\b(" + "|".join(k for k in _ORDINALS if not k[0].isdigit()) + r")\s+(?:question|one|pattern)\b",
    re.I)
# voice typing hears "word one" as "world one" / "would one" / "read one"
_WORD_SELECT = re.compile(r"\b(?:words?|world|would|work|write|writing|read)\s*(?:number\s*)?#?\s*"
                          r"(\d{1,2}|" + "|".join(_ORDINALS) + r")\b", re.I)


def _num(tok: str) -> Optional[int]:
    tok = tok.lower()
    if tok.isdigit():
        return int(tok)
    return _ORDINALS.get(tok)


def find_selection(kid_text: str, items: list[WorksheetItem]) -> Optional[WorksheetItem]:
    """'what is the fourth question' / 'question 4' / 'no 4' / 'word 2' -> that item."""
    t = kid_text or ""
    m = _WORD_SELECT.search(t)
    if m and any(i.kind == "words" for i in items):
        n = _num(m.group(1))
        return next((i for i in items if i.label == f"word {n}"), None)
    # "26b", "26 b", "26(b)", "question 26 b"
    for mm in re.finditer(r"\b(\d{1,2})\s*\(?\s*([a-h])\s*\)?(?![a-z])", t.lower()):
        hit = next((i for i in items if i.label == mm.group(1) + mm.group(2)), None)
        if hit:
            return hit
    m = _SELECT.search(t)
    if m:
        n = _num(m.group(1) or m.group(2))
        return next((i for i in items if i.label == str(n) or i.label == f"{n}a"), None)
    return None


def numbers_named(kid_text: str, items: list[WorksheetItem]) -> list[WorksheetItem]:
    """'check 21 22 23 24' / '22' -> the questions those numbers name (in order)."""
    out = []
    toks = []
    for tok in re.findall(r"(?<![\d.:])(\d{1,4})(?![\d:])", kid_text or ""):
        # voice typing glues "21 22" into "2122"
        toks += [tok[:2], tok[2:]] if len(tok) == 4 else ([tok] if len(tok) <= 2 else [])
    for n in toks:
        for i in items:
            if (i.label == n or re.fullmatch(n + r"[a-h]", i.label)) and i not in out:
                out.append(i)
    return out


def strip_selection(kid_text: str, label: str = "") -> str:
    """'word 2 - 375' -> ' - 375': what's left once the 'which question' part is
    removed, so an answer said in the same breath isn't mistaken for '2'.
    With the chosen label ('22', '26a') the bare number / '26 a' is removed too."""
    t = _WORD_SELECT.sub(" ", kid_text or "")
    t = _SELECT.sub(" ", t)
    m = re.fullmatch(r"(\d{1,2})([a-h]?)", label or "")
    if m:
        num, let = m.group(1), m.group(2)
        pat = r"(?<![\d])" + num + (r"\s*\(?\s*" + let + r"\s*\)?(?![a-z])" if let else r"(?![\d])")
        t = re.sub(pat, " ", t, count=1, flags=re.I)
    return t


def asked_for_number(kid_text: str) -> Optional[int]:
    """The question number the kid named, even if the page has no such question."""
    m = _SELECT.search(kid_text or "")
    return _num(m.group(1) or m.group(2)) if m else None


def page_summary(items: list[WorksheetItem]) -> str:
    pats = [i for i in items if i.kind == "pattern"]
    done = [i.label for i in pats if i.finished_on_paper]
    todo = [i.label for i in items if i.kind != "words" and not i.finished_on_paper]
    words = [i for i in items if i.kind == "words"]
    parts = []
    if pats or todo:
        parts.append(f"I can see {len([i for i in items if i.kind != 'words'])} questions on your page")
    if words:
        parts.append(f"and {len(words)} numbers to write")
    msg = " ".join(parts) + "! 📄"
    if done:
        msg += (f" Question {done[0]} already looks filled in — great work!" if len(done) == 1
                else f" Questions {_and_list(done)} already look filled in — great work!")
    for i in items:
        for v in i.suspect.values():
            msg += (f" In question {i.label}, {v} doesn't quite fit the pattern"
                    " (or maybe I misread your writing!) — we can check it together.")
    if todo:
        msg += (f" Question {todo[0]} is ready to do." if len(todo) == 1
                else f" Ready to do: {_and_list(todo)}.")
    msg += " Which one shall we do? Just say \"question " + (todo[0] if todo else "1") + "\""
    msg += (" — or \"word 1\" for the write-the-number ones." if words else ".")
    return msg


def _and_list(xs: list[str]) -> str:
    xs = list(xs)
    if len(xs) == 1:
        return xs[0]
    return ", ".join(xs[:-1]) + " and " + xs[-1]


def numbers_in_kid_text(t: str) -> list[int]:
    """Numbers a kid said or typed. Voice typing turns 'four fifty' into
    '4:50' and 'four hundred' into '400' - fold clock-looking tokens back."""
    t = re.sub(r"(?<!\d)(\d{1,2}):(\d{2})(?!\d)", r"\1\2", t or "")
    out = [int(x.replace(",", "")) for x in re.findall(r"(?<![\d.])\d[\d,]*", t)]
    if not out:
        for m in _WORD_NUM.finditer(t):
            v = words_to_int(m.group(0))
            if v:
                out.append(v)
        if not out:
            from geeta.leak_guard import spoken_numbers_in_text
            out = [int(v) for v in spoken_numbers_in_text(t) if float(v).is_integer()]
    return out


_ASK_QUESTION = ("what is the question", "what's the question", "whats the question",
                 "give me the question", "read the question", "read it", "say it again",
                 "repeat", "show me the question", "the question again", "what question",
                 "show the question", "tell me the question")
_STUCK = ("stuck", "don't know", "dont know", "do not know", "idk", "help", "no clue",
          "not sure", "confused", "hard", "hint", "how do i", "how do you")
_PRAISE = ["Yes! 🎉", "You got it! ⭐", "Brilliant! 🌟", "Spot on! 🙌", "Yay, that's right! 🎈"]


def _unit_hint(prev: int, step: int) -> str:
    """A concrete way for a 6-8 year old to make ONE jump. Never says where it lands."""
    a = abs(step)
    way = "up" if step > 0 else "back"
    if a % 100 == 0:
        return (f"Start at {prev} and count {way} in hundreds, {a // 100} "
                f"{'time' if a == 100 else 'times'}. Put up a finger for each hundred.")
    if a % 10 == 0:
        k = a // 10
        return (f"Start at {prev} and count {way} in tens — {k} "
                f"{'ten' if k == 1 else 'tens'}. Put up a finger for each ten you say.")
    if a <= 10:
        return f"Start at {prev} and count {way} {a} on your fingers. Where do you land?"
    return f"Start at {prev}. Count {way} {a // 10} tens, then {a % 10} more ones."


# Mistake categories kept for the parent report + the kid's learning profile.
# Short, parent-readable, no kid wording stored.
MISTAKE_TEXT = {
    "jump_finding": "finding how big the jump is",
    "jump_size": "using the wrong jump size",
    "direction": "going the wrong way in a pattern",
    "whole_number": "saying only the end of a big number (e.g. 40 for 840)",
    "zero_placeholder": "remembering the 0 when there are no tens (e.g. 205)",
    "tens_ones": "writing the tens-and-ones part",
    "hundreds": "finding the hundreds digit",
    "wrong_operation": "choosing between adding and taking away (read the story carefully)",
    "off_by_one": "counting carefully - answers were just 1 away",
    "tens_slip": "keeping track of the tens",
    "digits_swapped": "putting digits in the right place (tens vs ones)",
    "one_group_off": "counting the right number of groups",
    "wrong_part": "reading which part the question asks for",
    "share_too_many": "sharing - giving each one more than there is to share",
    "share_too_few": "sharing - stopping before everyone got as much as they could",
    "remainder_too_big": "remainders - the left-over must be smaller than the number of groups",
    "wrong_remainder": "working out what is left over after sharing",
}
# categories too vague to help a parent ("a wrong answer") - kept for stats, left out of reports
MISTAKE_QUIET = {"wrong_answer", "step_slip"}


def skill_label(item: "WorksheetItem") -> str:
    """Parent-readable skill name for one worksheet question."""
    if item.kind == "arith" and item.meta.get("problem") is not None:
        return item.meta["problem"].skill
    if item.kind == "pattern" and item.step:
        return (f"Number patterns: counting {'on' if item.step > 0 else 'back'} "
                f"by {abs(item.step)}s")
    if item.kind == "words":
        return "Writing numbers from words (hundreds, tens and ones)"
    if item.kind == "picture":
        h = (item.meta.get("heading") or item.text).lower()
        return ("Shapes: lines of symmetry" if "symmetr" in h else "Lines: parallel lines" if "parallel" in h
                else "Shapes: sides and corners" if re.search(r"segment|angle|side|corner", h) else "Drawing task")
    return "Homework question"


class _Tracks:
    def _miss(self, cat: str, detail: str = "") -> None:
        self.mistakes.append({"cat": cat, "detail": detail})


class PatternTutor(_Tracks):
    """Deterministic tutor for ONE number pattern. Every number it says is
    computed by Python from the kid's own page - never invented by the model -
    and it never says a missing number before the kid does."""

    def __init__(self, item: WorksheetItem):
        self.item = item
        self.full = fill_pattern(item.terms, item.step)
        self.shown = list(item.terms)          # gaps the kid has filled get written in
        self.jump_known = False
        self.hint_level = 0
        self.done = False
        self.wrong_in_a_row = 0
        self.mistakes: list[dict] = []

    # -- what the kid can see ------------------------------------------------
    def board(self) -> str:
        return ", ".join("___" if v is None else str(v) for v in self.shown)

    def _gap(self) -> Optional[int]:
        return next((i for i, v in enumerate(self.shown) if v is None), None)

    def _neighbour(self, g: int) -> tuple[int, int]:
        """Closest known number to gap g, and how many jumps away it is."""
        best = None
        for i, v in enumerate(self.shown):
            if v is not None and (best is None or abs(i - g) < abs(best[0] - g)
                                  or (abs(i - g) == abs(best[0] - g) and i < g)):
                best = (i, v)
        return best[1], g - best[0]

    def _jump_pair(self) -> tuple[int, int, int]:
        """Two known numbers to discover the jump from (adjacent if possible)."""
        known = [(i, v) for i, v in enumerate(self.item.terms) if v is not None]
        for (i, a), (j, b) in zip(known, known[1:]):
            if j == i + 1:
                return a, b, 1
        (i, a), (j, b) = known[0], known[1]
        return a, b, j - i

    def _ask_gap(self) -> str:
        g = self._gap()
        near, hops = self._neighbour(g)
        s = self.item.step
        if hops == 1:
            return f"What number comes straight after {near}?"
        if hops == -1:
            return f"What number comes just before {near}? (Go {abs(s)} back!)"
        return f"Let's fill the next gap. Start from {near} and do {abs(hops)} jumps of {abs(s)}."

    def _ask_jump(self) -> str:
        a, b, gap = self._jump_pair()
        if gap == 1:
            return (f"Let's find the secret jump first! 🐸 Look at {a} and {b} — they sit "
                    f"right next to each other. How much bigger is {b} than {a}?"
                    if b > a else
                    f"Let's find the secret jump first! 🐸 Look at {a} and {b} — they sit "
                    f"right next to each other. How much smaller is {b} than {a}?")
        return (f"Let's find the secret jump first! 🐸 From {a} to {b} there are {gap} jumps. "
                f"What does each jump add?")

    # -- turns ----------------------------------------------------------------
    def intro(self) -> str:
        n = len(self.item.gaps)
        odd = ""
        if self.item.suspect:
            v = next(iter(self.item.suspect.values()))
            odd = (f"I took out {v} because it doesn't fit the jumps (maybe I misread it!) — "
                   "let's work that spot out again. ")
        title = "Let's do it!" if self.item.label == "typed" else f"Question {self.item.label}!"
        return (f"{title} Here it is:\n{self.board()}\n{odd}"
                f"There {'is 1 gap' if n == 1 else f'are {n} gaps'} to fill. " + self._ask_jump())

    def restate(self) -> str:
        return (f"Sure! {'It' if self.item.label == 'typed' else 'Question ' + self.item.label} says:\n{self.board()}\n"
                + (self._ask_gap() if self.jump_known else self._ask_jump()))

    def hint(self) -> str:
        self.hint_level += 1
        s = self.item.step
        if not self.jump_known:
            a, b, gap = self._jump_pair()
            if self.hint_level == 1:
                if gap == 1 and abs(s) % 10 == 0:
                    return (f"Count from {a} to {b} in tens: {a + (10 if s > 0 else -10)}, "
                            f"{a + (20 if s > 0 else -20)}... How many tens did you count?")
                if gap == 1:
                    return (f"Count on from {a} until you reach {b}, using your fingers. "
                            "How many fingers are up?")
                return (f"From {a} to {b} is {abs(b - a)} altogether. Share that equally "
                        f"between the {gap} jumps — how much is each jump?")
            # the jump itself isn't the answer - after one hint we tell them
            self.jump_known = True
            return (f"Here's the secret: this pattern goes {'up' if s > 0 else 'down'} by "
                    f"{abs(s)} every time! " + self._ask_gap())
        g = self._gap()
        near, hops = self._neighbour(g)
        if self.hint_level <= 1:
            return (f"Every jump {'adds' if s > 0 else 'takes away'} {abs(s)}. "
                    f"What is {near} {'+' if s * hops > 0 else '−'} {abs(s)}?"
                    if abs(hops) == 1 else self._ask_gap())
        return _unit_hint(near, s if hops > 0 else -s)

    def _op(self, hops: int) -> str:
        return "+" if self.item.step * hops > 0 else "−"

    def take_numbers(self, nums: list[int], kid_text: str = "") -> str:
        s, a_s = self.item.step, abs(self.item.step)
        low = (kid_text or "").lower()
        said = list(nums)
        talks_jump = any(w in low for w in ("more", "jump", "less", "each time", "every time"))
        if not self.jump_known:
            a, b, gap = self._jump_pair()
            g = self._gap()
            if self.full[g] in nums:
                self.jump_known = True     # they skipped ahead and filled a gap: great
            elif a_s in nums:
                self.jump_known = True
                self.hint_level = 0
                self.wrong_in_a_row = 0
                return f"Yes! It jumps by {a_s} every time! 🐸 " + self._ask_gap()
            else:
                n = nums[0]
                lo = min(a, b)
                if n >= lo // 2 and n > 3 * abs(b - a) and not talks_jump:
                    # "7:55" while we're hunting the jump: that's a pattern
                    # number, not a jump - don't do "750 + 755" maths on it
                    return ("That looks like a number from the pattern! 🙂 First let's find "
                            f"the jump: how much do we add to get from {a} to {b}?"
                            if gap == 1 else
                            f"That looks like a number from the pattern! 🙂 First the jump: "
                            f"from {a} to {b} there are {gap} jumps. How big is each one?")
                self.wrong_in_a_row += 1
                self._miss("jump_finding", f"guessed {n} for a jump of {a_s}")
                if self.wrong_in_a_row >= 3:
                    return self.hint()
                if gap == 1:
                    land = a + n if b > a else a - n
                    if n < abs(b - a):
                        return (f"Good try! But {a} {'+' if b > a else '−'} {n} is only {land}, "
                                f"not {b}. The jump is bigger! Try again.")
                    return (f"Ooh, nearly! {a} {'+' if b > a else '−'} {n} would be {land} — "
                            f"that goes past {b}. Try a smaller jump.")
                land = a + n * gap if b > a else a - n * gap
                return (f"Let's test it! If each jump was {n}, then {gap} jumps from {a} would "
                        f"land on {land} — but we need {b}. "
                        + ("Try a smaller jump." if abs(land - a) > abs(b - a) else "Try a bigger jump."))

        g = self._gap()
        if g is None:
            return f"This one's all done:\n{self.board()}\nWant to pick another question? 😊"
        exp = self.full[g]
        if len(nums) > 1:
            # "50 450" / "820 825 830 835": start from the number that fits the
            # gap, and drop the jump size the kid said out loud on the way
            # voice typing glues counting together: "820, 825" -> "82825"
            nums = [exp if x != exp and str(x).endswith(str(exp)) else x for x in nums]
            if exp in nums:
                nums = nums[nums.index(exp):]
            else:
                nums = [x for x in nums if x != a_s and x not in self.shown] or nums[:1]
        replies = []
        for n in nums:
            g = self._gap()
            if g is None:
                break
            exp = self.full[g]
            if n == exp:
                self.shown[g] = n
                self.wrong_in_a_row = 0
                self.hint_level = 0
                replies.append(n)
                continue
            if replies:
                if n in self.shown:
                    continue       # a printed number said while counting along
                break              # stop at the first slip; ask for the next gap below
            near, hops = self._neighbour(g)
            if n in self.shown:
                return f"Yes, {n} is already on the page! Now the gap. " + self._ask_gap()
            if n in self.full:
                return (f"Ooh, {n} IS in this pattern — but it goes in a later gap. "
                        + self._ask_gap())
            # "40" for 840: the kid said only the end of the number
            if n < 100 <= abs(exp) and abs(exp) % 100 == n:
                self._miss("whole_number", f"said {n} for {exp}")
                return ("Ooh, the end part is right! 👍 Now say the WHOLE number — "
                        "with the hundreds at the front.")
            # "is it 10 more or 20 more?" - they're asking about the jump
            if talks_jump or (n < 100 <= abs(near)):
                if a_s in said:
                    return (f"Yes — every jump is {a_s} more! So what is "
                            f"{near} {self._op(hops)} {a_s}?")
                return (f"This pattern jumps {a_s} each time, not {n}. "
                        f"What is {near} {self._op(hops)} {a_s}?")
            # wrong: say what the kid's number actually did, then nudge
            self.wrong_in_a_row += 1
            if abs(hops) == 1:
                did = n - near
                if (did > 0) != ((s * hops) > 0):
                    self._miss("direction")
                    msg = (f"Hmm, look — this pattern goes {'up' if s > 0 else 'down'}, so the "
                           f"number after {near} must be {'bigger' if s * hops > 0 else 'smaller'}. "
                           "Try again!")
                else:
                    self._miss("jump_size", f"jumped {abs(did)} instead of {a_s}")
                    msg = (f"Not quite — {near} to {n} is a jump of {abs(did)}, but our "
                           f"jump is {a_s}. Try one jump of {a_s} from {near}. 💪")
                if self.wrong_in_a_row >= 2:
                    msg += " " + _unit_hint(near, s * hops)
                return msg
            return "Not quite, that's okay! " + self._ask_gap()
        praise = (f"{random.choice(_PRAISE)} {replies[0]} is right!" if len(replies) == 1 else
                  f"{random.choice(_PRAISE)} {_and_list([str(r) for r in replies])} are {'both' if len(replies) == 2 else 'all'} right!")
        if self._gap() is None:
            self.done = True
            return (praise + f"\nYou filled the whole pattern:\n{self.board()}\n"
                    "YOU did that! 🏆 Want to pick another question?")
        return praise + " " + self._ask_gap()

    def nudge(self) -> str:
        """Short, quiet check-in - the kid may just be busy writing on paper,
        so don't re-read the whole question at them."""
        ask = self._ask_gap() if self.jump_known else "What's the jump?"
        return random.choice(["Take your time! 😊 ", "No rush! ", "I'm here when you're ready. "]) + ask

    def turn(self, kid_text: str) -> str:
        low = (kid_text or "").lower()
        if any(k in low for k in _ASK_QUESTION):
            return self.restate()
        nums = numbers_in_kid_text(kid_text)
        if nums:
            return self.take_numbers(nums, kid_text)
        if any(k in low for k in _STUCK):
            return self.hint()
        # no number, not stuck, not asking for the question: a short nudge -
        # re-reading the whole board every time is what kids found annoying
        return self.nudge()


class WordNumberTutor(_Tracks):
    """'four hundred and eighty-five' -> write it in digits."""

    def __init__(self, item: WorksheetItem):
        self.item = item
        self.ans = item.answer
        self.h, self.rest = divmod(self.ans, 100)
        self.hint_level = 0
        self.done = False
        self.got_hundreds = False
        self.mistakes: list[dict] = []

    def intro(self) -> str:
        return (f"Let's write this one in numbers: \"{self.item.text}\". ✏️ "
                "Start with the hundreds. How many hundreds is it?")

    def nudge(self) -> str:
        return f"Take your time! 😊 What's \"{self.item.text}\" in numbers?"

    def restate(self) -> str:
        return f"Sure! Write \"{self.item.text}\" using digits. What do you think?"

    def hint(self) -> str:
        self.hint_level += 1
        words = self.item.text.split(" and ")
        if not self.got_hundreds:
            return f"Listen for the word before \"hundred\": \"{words[0]}\". That digit goes first!"
        tail = words[1] if len(words) > 1 else ""
        if self.rest < 10:
            return (f"After the hundreds comes \"{tail}\". That's less than ten, so the tens "
                    "place needs a 0 to hold its spot. Try writing the whole number.")
        return f"Now the \"{tail}\" part — write it as a 2-digit number after the hundreds digit."

    def turn(self, kid_text: str) -> str:
        low = (kid_text or "").lower()
        if any(k in low for k in _ASK_QUESTION):
            return self.restate()
        nums = numbers_in_kid_text(kid_text)
        if not nums:
            if any(k in low for k in _STUCK):
                return self.hint()
            return self.restate()
        n = nums[0]
        if n == self.ans:
            self.done = True
            return f"{random.choice(_PRAISE)} {n} is exactly right! Want to do another one?"
        if not self.got_hundreds and n == self.h:
            self.got_hundreds = True
            return (f"Yes, {self.h} hundreds! Now the \"{self.item.text.split(' and ')[-1]}\" "
                    "part. Can you write the whole number?")
        if self.rest < 10 and n == self.h * 10 + self.rest:
            self._miss("zero_placeholder", f"wrote {n} for {self.ans}")
            return ("So close! But we need to show there are NO tens. "
                    "Pop a 0 in the middle to keep the tens place. Try again!")
        if n // 100 == self.h and n != self.ans:
            self.got_hundreds = True
            self._miss("tens_ones", f"wrote {n} for {self.ans}")
            return (f"Great start — the {self.h} hundreds is right! "
                    f"Check the \"{self.item.text.split(' and ')[-1]}\" part again.")
        if n >= 100 and n // 100 != self.h:
            self._miss("hundreds", f"wrote {n} for {self.ans}")
        return "Not quite, that's okay! " + self.hint()


class PictureTutor(_Tracks):
    """A question about a picture (draw it, lines of symmetry, parallel lines, count the sides).
    GEETA can't see the picture well enough to check it, so it coaches honestly - it never
    pretends to mark it and never says how many lines/sides there are."""

    def __init__(self, item: WorksheetItem):
        self.item = item
        self.done = False
        self.unchecked = False
        self.hint_level = 0
        self.mistakes: list[dict] = []
        self.counts: list[int] = []
        h = ((item.meta.get("heading") or item.text) + " " + (item.meta.get("asks") or "")).lower()
        self.topic = ("symmetry" if "symmetr" in h else "parallel" if "parallel" in h
                      else "count" if re.search(r"segments?|angles?|\bsides\b|corners|vertices", h) else "draw")

    def _task(self) -> str:
        return f"Question {self.item.label}: {self.item.meta.get('heading') or self.item.text}"

    def _guide(self) -> str:
        return {
            "symmetry": "A line of symmetry is a fold line - if you folded the shape there, both halves would match "
                        "exactly, like a butterfly. 🦋 Where could you fold this one so both sides match? Draw that line. "
                        "Some shapes have more than one - can you find another?",
            "parallel": "Parallel lines are like train tracks 🚂 - they go the same way and never meet. Trace those two "
                        "with your blue crayon. Then draw a red line that crosses over BOTH of them.",
            "count": "A segment is one straight side, and an angle is a corner where two sides meet. Put your finger "
                     "on one corner and go all the way round, counting each straight side. Then count the corners. "
                     "How many segments, and how many angles?",
            "draw": "Draw it on your paper - or on the board here. Count as you draw so you get the right number!",
        }[self.topic]

    def intro(self) -> str:
        if self.topic == "draw":
            return f"{self._task()} ✏️ A drawing one! {self._guide()} Tell me when you've finished."
        return (f"{self._task()}\nThis one is about the picture on your page - I can't see pictures well enough "
                f"to check them, so you're the boss! {self._guide()}")

    def restate(self) -> str:
        return self.intro()

    def explain(self) -> str:
        return self.intro()

    def nudge(self) -> str:
        return f"Take your time! 😊 {self._guide()}"

    def hint(self) -> str:
        self.hint_level += 1
        return {
            "symmetry": "Try this: imagine a mirror standing on your line. Does the shape look exactly the same on both "
                        "sides? Try a line straight up and down, then one side to side, then corner to corner.",
            "parallel": "Look for two lines that stay the same distance apart all the way along - they never get closer.",
            "count": "Put a little dot on each corner as you count, so you don't count one twice. Then count the "
                     "straight lines between the dots.",
            "draw": "Start with one, then count it. Then draw the next one and count again!",
        }[self.topic]

    def _finish(self, lead: str) -> str:
        self.done = True
        self.unchecked = True            # GEETA can't check pictures - never counted as "solved"
        return (f"{lead} I can't see your page to mark it, so show your grown-up when you're done. "
                "Want to do another question?")

    def turn(self, kid_text: str) -> str:
        low = (kid_text or "").lower()
        if any(k in low for k in _ASK_QUESTION) or re.search(r"explain|don'?t understand|do not understand", low):
            return self.restate()
        if any(k in low for k in _STUCK) or re.search(r"\bhow\b|\bwhat is\b|\bwhat's\b", low):
            if self.topic == "count" and re.search(r"segment", low):
                return "A segment is just one straight side of the shape. " + self.hint()
            if self.topic == "count" and re.search(r"angle", low):
                return "An angle is a corner - where two straight sides meet. " + self.hint()
            return self.hint()
        if self.topic == "count":
            nums = numbers_in_kid_text(kid_text)
            self.counts += nums
            if len(self.counts) >= 2:
                a, b = self.counts[-2], self.counts[-1]
                if a != b:
                    self.counts = []
                    return ("Hmm, on a closed shape like this the number of sides and corners is always the same - every "
                            "corner joins two sides. Count both again with your finger!")
                return self._finish(f"Great counting! 🌟 {a} segments and {b} angles - they match, which is a good sign.")
            if len(self.counts) == 1:
                return f"{self.counts[0]} - nice! Now count the other one: segments are the sides, angles are the corners."
            return self.nudge()
        if re.search(r"\b(done|finished|drew|drawn|did it|i did|yes|yep|ok|okay|ready)\b", low):
            return self._finish("Awesome! ✏️")
        return self.nudge()


def make_tutor(item: WorksheetItem, buddy: str = "GEETA"):
    if item.kind == "picture":
        return PictureTutor(item)
    if item.kind == "pattern":
        return PatternTutor(item)
    if item.kind == "words":
        return WordNumberTutor(item)
    if item.kind == "arith" and item.meta.get("problem") is not None:
        from geeta.arith import ArithTutor
        return ArithTutor(item, item.meta["problem"], buddy=buddy)
    return None
